# NBA 3.1 Model Comparison: one model vs two, Poisson vs Tweedie

Runs after the three `03_nba_3_1_model_based_selection` tasks (segments `all`, `writers`, `non_writers`) and compares them on the **same test rows** (c25-c26, every HCP):
* **One model:** the `all` model ranks every HCP.
* **Two models:** the `writers` and `non_writers` predictions are put into one ranking (both predict NBRx counts, so they are on the same scale).
* Each approach with the **Poisson** and the **Tweedie** loss.

Main metric (call notes, Raman): share of **new writers** (non-writer HCP-months with an NBRx) in the top 5 / 10 / 20% of the ranking, while the share of **existing-writer NBRx** in the top does not drop. Lift is shown against the one-model Poisson version. Also: overlap of the selected feature lists. Outputs in `model_based/comparison/` and an MLflow run `comparison`.

In [0]:
import numpy as np, pandas as pd, json, os
import mlflow

dbutils.widgets.text("RUN_NAME", "combined_2026_11_26_full")
dbutils.widgets.text("SAMPLE_NAME", "all_hcps_5_train_months")
dbutils.widgets.text("STAT_VERSION", "statistical_v2")
dbutils.widgets.text("MAX_FEATURES", "0")
dbutils.widgets.dropdown("WRITE_OUTPUT", "false", ["true", "false"])
RUN_NAME     = dbutils.widgets.get("RUN_NAME").strip()
SAMPLE_NAME  = dbutils.widgets.get("SAMPLE_NAME").strip()
STAT_VERSION = dbutils.widgets.get("STAT_VERSION").strip().strip("/")
MAX_FEATURES = int(dbutils.widgets.get("MAX_FEATURES") or 0)
WRITE_OUTPUT = dbutils.widgets.get("WRITE_OUTPUT") == "true"

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
IS_TEST = bool(MAX_FEATURES) or "_test_" in STAT_VERSION   # same rule as notebook 03
MB_DIR  = f"{DEV}{RUN_NAME}/feature_selection/_sample_{SAMPLE_NAME}/model_based/" + (f"_test_{MAX_FEATURES}_features/" if IS_TEST else "")
OUT_DIR = MB_DIR + "comparison/"
EXPERIMENT = "/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_model_based_selection"
SEGMENTS   = ["all", "writers", "non_writers"]
LOSSES     = ["poisson", "tweedie"]
TOP_SHARES = [0.05, 0.10, 0.20]

preds = {s: pd.read_parquet(f"{MB_DIR}{s}/test_predictions.parquet") for s in SEGMENTS}
summaries = {s: json.load(open(f"{MB_DIR}{s}/run_summary.json")) for s in SEGMENTS}
sel = {(s, l): set(pd.read_csv(f"{MB_DIR}{s}/selected_features_{l}.csv").FEATURE) for s in SEGMENTS for l in LOSSES}

# The writer and non-writer rows must together be exactly the rows of the all-HCP model
w_ids, n_ids, a_ids = (set(preds[s].HCP_COHORT_ID) for s in ["writers", "non_writers", "all"])
assert not w_ids & n_ids and (w_ids | n_ids) == a_ids, "segment test rows do not add up to the all-HCP rows"
assert preds["writers"].IS_EXISTING_WRITER.all() and not preds["non_writers"].IS_EXISTING_WRITER.any()

base = preds["all"][["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "IS_EXISTING_WRITER", "TARGET_NBRX"]]
base = base.merge(preds["all"][["HCP_COHORT_ID"] + [f"PRED_{l.upper()}" for l in LOSSES]]
                  .rename(columns={f"PRED_{l.upper()}": f"ONE_MODEL_{l.upper()}" for l in LOSSES}), on="HCP_COHORT_ID")
two = pd.concat([preds["writers"], preds["non_writers"]])[["HCP_COHORT_ID"] + [f"PRED_{l.upper()}" for l in LOSSES]]
base = base.merge(two.rename(columns={f"PRED_{l.upper()}": f"TWO_MODELS_{l.upper()}" for l in LOSSES}), on="HCP_COHORT_ID")
assert len(base) == len(a_ids) and base.notna().all().all()
APPROACHES = [f"{a}_{l.upper()}" for a in ["ONE_MODEL", "TWO_MODELS"] for l in LOSSES]
print(f"{len(base):,} test rows, {base.PROVIDER_ID.nunique():,} HCPs; approaches: {APPROACHES}")
display(pd.DataFrame([{"segment": s, "candidates": summaries[s]["candidates"], "train_rows": summaries[s]["train_rows"],
                       "test_rows": summaries[s]["test_rows"], "tweedie_power": summaries[s]["best_tweedie_power"],
                       **{f"selected_{l}": summaries[s]["selected"][l] for l in LOSSES}} for s in SEGMENTS]))

In [0]:
def capture(df, pred_col):
    """Same definition as notebook 03: HCPs ranked within each test month; shares in the top."""
    writer = df.IS_EXISTING_WRITER.astype(bool).values
    new_writer = ~writer & (df.TARGET_NBRX.values > 0)
    yv = df.TARGET_NBRX.values
    rank = df.groupby("COHORT_ID")[pred_col].rank(ascending=False, method="first").values
    size = df.groupby("COHORT_ID")[pred_col].transform("size").values
    out = []
    for s in TOP_SHARES:
        top = rank <= np.ceil(size * s)
        out.append({"approach": pred_col, "top_share": s, "new_writers": int(new_writer.sum()),
                    "new_writers_in_top": int((new_writer & top).sum()),
                    "new_writer_capture": (new_writer & top).sum() / new_writer.sum(),
                    "writer_nbrx_capture": yv[writer & top].sum() / yv[writer].sum(),
                    "nbrx_capture": yv[top].sum() / yv.sum()})
    return out

comparison = pd.DataFrame([r for a in APPROACHES for r in capture(base, a)])
ref = comparison[comparison.approach == "ONE_MODEL_POISSON"].set_index("top_share")
for m in ["new_writer_capture", "writer_nbrx_capture", "nbrx_capture"]:
    comparison[f"{m}_lift_vs_one_poisson"] = comparison[m] / comparison.top_share.map(ref[m]) - 1
comparison["keeps_writers"] = comparison.writer_nbrx_capture >= comparison.top_share.map(ref.writer_nbrx_capture) - 1e-12
display(comparison)

# Within-segment results reported by each model task (each segment ranked on its own)
segment_metrics = pd.concat([pd.read_csv(f"{MB_DIR}{s}/test_metrics.csv").assign(segment=s) for s in SEGMENTS], ignore_index=True)
display(segment_metrics)

In [0]:
overlap = []
for l in LOSSES:
    a, wr, nw = sel[("all", l)], sel[("writers", l)], sel[("non_writers", l)]
    overlap.append({"loss": l, "all": len(a), "writers": len(wr), "non_writers": len(nw),
                    "writers_and_non_writers": len(wr & nw), "writers_or_non_writers": len(wr | nw),
                    "all_and_two_model_union": len(a & (wr | nw)), "only_all": len(a - (wr | nw)),
                    "only_two_models": len((wr | nw) - a)})
for s in SEGMENTS:
    overlap.append({"loss": f"poisson vs tweedie ({s})", "all": len(sel[(s, 'poisson')] & sel[(s, 'tweedie')])})
overlap = pd.DataFrame(overlap)
display(overlap)

In [0]:
# Independent check: the one-model Poisson top-10% new-writer capture recomputed by sorting each month
chk = []
for _, g in base.groupby("COHORT_ID"):
    chk.append(g.sort_values("ONE_MODEL_POISSON", ascending=False, kind="mergesort").head(int(np.ceil(len(g) * 0.10))))
chk = pd.concat(chk)
nw_top = ((~chk.IS_EXISTING_WRITER) & (chk.TARGET_NBRX > 0)).sum()
nw_all = ((~base.IS_EXISTING_WRITER) & (base.TARGET_NBRX > 0)).sum()
assert np.isclose(nw_top / nw_all, ref.loc[0.10, "new_writer_capture"]), "capture check failed"
print("checks passed")

best = comparison[comparison.top_share == 0.10].sort_values("new_writer_capture", ascending=False)
print("Top 10% ranking, best new-writer capture first:")
display(best[["approach", "new_writer_capture", "new_writer_capture_lift_vs_one_poisson", "writer_nbrx_capture", "keeps_writers"]])

if WRITE_OUTPUT:
    os.makedirs(OUT_DIR, exist_ok=True)
    comparison.to_csv(OUT_DIR + "comparison.csv", index=False)
    segment_metrics.to_csv(OUT_DIR + "segment_metrics.csv", index=False)
    overlap.to_csv(OUT_DIR + "feature_overlap.csv", index=False)
    mlflow.set_experiment(EXPERIMENT)
    with mlflow.start_run(run_name="comparison" + (" (test)" if IS_TEST else "")):
        mlflow.set_tags({"step": "model_comparison", "is_test": str(IS_TEST), "stat_version": STAT_VERSION})
        for _, r in comparison.iterrows():
            s = int(round(r.top_share * 100))
            mlflow.log_metric(f"{r.approach.lower()}_new_writer_capture_top{s}", float(r.new_writer_capture))
            mlflow.log_metric(f"{r.approach.lower()}_writer_nbrx_capture_top{s}", float(r.writer_nbrx_capture))
        for fn in ["comparison.csv", "segment_metrics.csv", "feature_overlap.csv"]:
            mlflow.log_artifact(OUT_DIR + fn)
    print("Saved to", OUT_DIR, sorted(os.listdir(OUT_DIR)))
else:
    print("WRITE_OUTPUT = false: nothing saved")